In [ ]:
import requests
import json

# ========== 配置区域 ==========
node_ip = "ai.catarc.ac.cn"
port = "9080"
app_id = "app-ewb874zb"           
api_key = "75f8f1988e58589446f29492680dcc78"     

# 是否启用详细调试（打印原始返回）
DEBUG = True
# =============================

# 构造请求 URL
url = f"http://{node_ip}:{port}/appforge/openapi/v1/InvokeApp/{app_id}"

# 请求头
headers = {
    "Authorization": f"Bearer {api_key}",  # 文档要求 Bearer 方式
    "Content-Type": "application/json",
    "Accept": "*/*",
    "Connection": "keep-alive"
}

# 测试请求体
payload = {
    "id": app_id,
    "query": "TARA中资产识别的输入输出是什么？损害场景的的输入输出是什么？威胁场景的输入输出是什么？攻击路径的输入输出是什么？"
}

print(">>> 正在测试智能体调用...")
print(f"URL: {url}")
print(f"Payload: {json.dumps(payload, ensure_ascii=False)}\n")

try:
    # 使用 stream=True 接收可能的 SSE 流式返回
    response = requests.post(url, headers=headers, json=payload, stream=True, timeout=30)

    print(f"HTTP 状态码: {response.status_code}")

    if response.status_code == 200:
        print("✅ 智能体调用成功，开始接收返回数据...\n")

        # 处理 SSE 返回：每行以 "data:" 开头
        for line in response.iter_lines():
            if not line:
                continue
            decoded = line.decode("utf-8")

            if DEBUG:
                print("原始行:", decoded)

            # 尝试解析 JSON
            if decoded.startswith("data:"):
                json_str = decoded[5:].strip()  # 去掉 "data:"
                if json_str:
                    try:
                        data = json.loads(json_str)
                        # 打印解析后的 JSON（美化）
                        print(json.dumps(data, indent=2, ensure_ascii=False))
                    except json.JSONDecodeError:
                        print("JSON 解析失败:", json_str)

    elif response.status_code == 401:
        print("❌ 认证失败，API Key 无效（HTTP 401）")
        print("返回内容：", response.text)
    elif response.status_code == 404:
        print("❌ 资源不存在（HTTP 404），请确认应用 ID 和 URL 是否正确")
        print("返回内容：", response.text)
    else:
        print(f"❌ 请求失败，状态码: {response.status_code}")
        print("返回内容：", response.text)

except requests.exceptions.RequestException as e:
    print("❌ 请求发生异常：", e)

>>> 正在测试智能体调用...
URL: http://ai.catarc.ac.cn:9080/appforge/openapi/v1/InvokeApp/app-ewb874zb
Payload: {"id": "app-ewb874zb", "query": "TARA中资产识别的输入输出是什么？损害场景的的输入输出是什么？威胁场景的输入输出是什么？攻击路径的输入输出是什么？"}

HTTP 状态码: 200
✅ 智能体调用成功，开始接收返回数据...

原始行: data: {"type":"knowledge","content":"[{\"chunk_id\":\"documentsegment-qjnuhncn\",\"dataset_id\":\"dataset-yfcpvf0m\",\"document_id\":\"document-s9zsdj0c\",\"document_name\":\"EE架构.xlsx\",\"content\":\"{\\\"序号\\\":\\\"13\\\",\\\"起始资产\\\":\\\"T-003\\\",\\\"可达资产\\\":\\\"T-003, K-008, T-001, T-002, T-010, Z-001, Z-002, T-004\\\"}\",\"content_shot\":\"{\\\"序号\\\":\\\"13\\\",\\\"起始资产\\\":\\\"T-003\\\",\\\"可达资产\\\":\\\"T-003, K-008, T-001, T-002, T-010, Z-001, Z-002, T-004\\\"}\",\"score\":0.4754182,\"bbox\":{},\"page_num\":\"-1\",\"file_id\":\"document-s9zsdj0c\"},{\"chunk_id\":\"documentsegment-ug2k65un\",\"dataset_id\":\"dataset-yfcpvf0m\",\"document_id\":\"document-s9zsdj0c\",\"document_name\":\"EE架构.xlsx\",\"content\":\"{\\\"序号\\\":\\\"14\\\",\\\"起始资产\\

In [5]:
import json

def parse_appforge_sse_line(line):
    """
    解析标准的 SSE (Server-Sent Events) 格式的一行数据
    """
    # 去除首尾空白符
    line = line.strip()
    
    # 如果是空行，通常表示一个事件结束，返回None
    if not line:
        return None
        
    result = {}
    
    # 标准SSE格式通常是 "field: value"
    if line.startswith("data:"):
        # 提取 data 后面的内容
        data_str = line[5:].strip()
        
        # 尝试将 data 解析为 JSON (很多SSE服务返回的是JSON)
        try:
            result["data"] = json.loads(data_str)
        except json.JSONDecodeError:
            # 如果不是JSON，直接作为字符串返回
            result["data"] = data_str
            
    elif line.startswith("event:"):
        result["event"] = line[6:].strip()
        
    elif line.startswith("id:"):
        result["id"] = line[3:].strip()
        
    else:
        # 其他情况直接作为原始数据
        result["raw"] = line
        
    return result
import requests
import json
from pprint import pprint

# ========== 配置区域 ==========
node_ip = "ai.catarc.ac.cn"
port = "9080"
app_id = "app-ewb874zb"           # 应用 ID
api_key = "75f8f1988e58589446f29492680dcc78"     # 替换真实 API Key

# ========== 构造请求 ==========
url = f"http://{node_ip}:{port}/appforge/openapi/v1/InvokeApp/{app_id}"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
    "Accept": "*/*",
    "Connection": "keep-alive"
}

payload = {
    "id": app_id,
    "query": "TARA中资产识别的输入输出是什么？损害场景的的输入输出是什么？威胁场景的输入输出是什么？攻击路径的输入输出是什么？"
}

# ========== 发起请求 ==========
print(">>> 正在测试智能体调用...")
print(f"URL: {url}")
print(f"Payload: {json.dumps(payload, ensure_ascii=False)}\n")

response = requests.post(url, headers=headers, json=payload, stream=True, timeout=30)
print(f"HTTP 状态码: {response.status_code}")

if response.status_code != 200:
    print("返回内容：", response.text)
    # 可以直接在这里结束，或者抛异常
else:
    # 存储解析后的结果
    events = []            # 所有事件
    knowledge_chunks = []  # type="knowledge" 的知识库片段
    answer_parts = []      # type="answer" 的文本片段
    answer_text = ""       # 最终回答
    done_event = None      # type="done" 的结束事件

    for raw_line in response.iter_lines():
        if not raw_line:
            continue
        decoded = raw_line.decode("utf-8")
        result = parse_appforge_sse_line(decoded)
        events.append(result)

        data = result.get("data")
        if not data:
            continue

        event_type = data.get("type")

        if event_type == "knowledge":
            # 知识库检索结果
            content_str = data.get("content", "")
            # content 本身是一个 JSON 字符串，需要再解析一次
            try:
                chunks = json.loads(content_str)
                if isinstance(chunks, list):
                    knowledge_chunks.extend(chunks)
                else:
                    knowledge_chunks.append(chunks)
            except json.JSONDecodeError:
                # 解析失败就直接存原字符串
                knowledge_chunks.append(content_str)

        elif event_type == "answer":
            # 流式回答片段
            text = data.get("content", "")
            answer_parts.append(text)
            answer_text += text  # 拼接完整回答

        elif event_type == "done":
            # 会话结束事件
            done_event = data

    # ========== 输出关键结果 ==========
    print("\n===== 解析后的结果 =====\n")

    print("【最终回答】")
    print(answer_text or "（无文本回答）\n")

    print("【知识库片段数量】", len(knowledge_chunks))
    if knowledge_chunks:
        print("第一个知识库片段示例：")
        pprint(knowledge_chunks[0])

    print("\n【会话信息】")
    if done_event:
        print("conversation_id:", done_event.get("conversation_id"))
        print("message_id:", done_event.get("message_id"))
        print("total_time:", done_event.get("total_time"))
    else:
        print("未找到 type=done 事件")

    # 如果你想看所有事件，可以取消下面的注释
    # print("\n【所有事件】")
    # for ev in events:
    #     print(ev)


>>> 正在测试智能体调用...
URL: http://ai.catarc.ac.cn:9080/appforge/openapi/v1/InvokeApp/app-ewb874zb
Payload: {"id": "app-ewb874zb", "query": "TARA中资产识别的输入输出是什么？损害场景的的输入输出是什么？威胁场景的输入输出是什么？攻击路径的输入输出是什么？"}

HTTP 状态码: 200

===== 解析后的结果 =====

【最终回答】


<用户问题分析>
用户询问TARA（威胁分析与风险评估）中资产识别、损害场景、威胁场景及攻击路径四个环节的输入输出定义。需严格依据知识库内容进行回答，未提及部分需明确说明无法提供准确信息。
</用户问题分析>

<知识库调用情况>
知识库包含EE架构.xlsx文档中6个数据片段，主要描述资产间的可达关系（如起始资产与可达资产列表），但未明确说明资产识别、损害场景、威胁场景的输入输出定义。仅攻击路径相关内容可通过"起始资产"和"可达资产"字段推断。
</知识库调用情况>

<最终回答>
攻击路径的输入为**起始资产**（如T-003<sup score="0.477469" fileid="document-s9zsdj0c" filename="EE架构.xlsx" chunkid="documentsegment-qjnuhncn" dsid="dataset-yfcpvf0m">1</sup>），输出为**可达资产列表**（如T-003, K-008, T-001等<sup score="0.477469" fileid="document-s9zsdj0c" filename="EE架构.xlsx" chunkid="documentsegment-qjnuhncn" dsid="dataset-yfcpvf0m">1</sup>）。

资产识别、损害场景、威胁场景的输入输出定义在提供的知识库中**未找到有效信息**，无法提供准确描述。
【知识库片段数量】 6
第一个知识库片段示例：
{'bbox': {},
 'chunk_id': 'documentsegment-qjnuhncn',
 'content': '{"序号":"13","起始资产":"T-003","可达资产":"T-

In [24]:
import requests
import json
import time
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Any, Optional
import pandas as pd  # 用于保存 Excel 结果
# ================= 1. 配置区域 =================
CONFIG = {
    "node_ip": "ai.catarc.ac.cn",
    "port": "9080",
    "app_id": "app-ewb874zb",
    "api_key": "75f8f1988e58589446f29492680dcc78",  # 请替换为真实的 API Key
    "timeout": 60,     # 单次请求超时时间（秒）
    "max_workers": 5   # 并发线程数（建议先设小一点测试，稳定后可调至 10-20）
}
# ================= 2. 提示词模板 =================
# 将用户提供的长提示词定义为常量
SYSTEM_PROMPT = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 标准。你的任务是根据给定的资产或系统上下文，执行威胁分析和风险评估（TARA），输出损害场景（危害场景）、威胁场景和攻击路径。
### 核心约束条件
1. **定义合规性**：所有输出必须严格遵循以下 ISO 21434 定义：
   - **危害场景**：必须包含[相关项功能与不良后果的关系]、[对道路使用者的危害]、[相关资产]。
   - **威胁场景**：必须包含[目标资产]、[资产被破坏的信息安全属性（保密性/完整性/可用性）]、[破坏原因/攻击意图]。
   - **攻击路径**：必须包含逻辑连贯的步骤，展示如何从攻击面到达目标资产以实现威胁场景。
2. **一致性**：威胁场景必须能直接导致危害场景；攻击路径必须能实现威胁场景。
3. **格式要求**：直接输出最后的 JSON 格式结构化结果，不要输出思考过程。
### 思维链指引
在生成最终结果之前，先进行以下逐步思考：
1. **资产分析**：分析输入中提到的资产及其关键信息安全属性（CIA）。
2. **危害推演**：如果该资产的信息安全属性被破坏，会对道路使用者造成什么具体的危害？。
3. **威胁构建**：谁（攻击者）可能利用什么弱点来破坏该资产的属性？构建具体的威胁场景描述。
4. **路径分析**：攻击者如何进入系统？需要经过哪些组件或层级？。详细拆解攻击步骤。
### 一、必须遵守的标准与定义（严格执行）
#### 1. 资产类别与安全属性（严格遵循）
- **资产类别 = 「数据」**：安全属性限定为：完整性、机密性、可用性。
- **资产类别 = 「信号」**：安全属性限定为：完整性、机密性、真实性、可用性。
- **资产类别 = 「部件」**：安全属性限定为：完整性、机密性、真实性、不可抵赖性、权限属性、可用性。
#### 2. 威胁类型与安全属性映射（严格遵循）
- **真实性** → 威胁类型为「欺骗」
- **完整性** → 威胁类型为「篡改」
- **不可抵赖性** → 威胁类型为「否认」
- **机密性** → 威胁类型为「信息泄露」
- **可用性** → 威胁类型为「拒绝服务」
- **权限属性** → 威胁类型为「提权」
#### 3. 核心场景定义（严格遵循 ISO 21434 第 15 章）
- **危害场景（损害场景）**：必须包含三个要素。
- **威胁场景**：必须包含三个要素。
- **攻击路径**：必须包含逻辑连贯的步骤，必须明确攻击的起点（如蜂窝网络、蓝牙、OBD、USB、物理接口）。
### 五、当前任务要求
- 每次仅输出一个 JSON 对象，不要有多余文字。
- 生成的样本应在风格、用词、结构与「种子示例」保持高度一致。
"""
# ================= 3. 核心功能函数 =================
def parse_appforge_sse_line(raw_line: str) -> Dict[str, Any]:
    """解析 AppForge SSE 协议的单行数据"""
    line = raw_line.strip()
    if not line or line.startswith(":"):
        return {}
    if line.startswith("data:"):
        data_str = line[len("data:"):].strip()
        try:
            return {"data": json.loads(data_str)}
        except json.JSONDecodeError:
            return {}
    return {}
def extract_json_from_text(text: str) -> Optional[Dict]:
    """从回答文本中提取 JSON 对象（处理可能存在的 markdown 格式）"""
    if not text:
        return None
    # 尝试直接解析
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # 尝试提取 ```json ... ``` 块
    json_match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass
    # 尝试提取纯花括号内容
    brace_match = re.search(r'\{.*\}', text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass
    return None
def invoke_tara_agent(asset_info: Dict[str, str], session: requests.Session) -> Dict[str, Any]:
    """
    调用智能体生成单个资产的 TARA 分析
    asset_info: 包含 '功能', '资产', '资产类别', '安全属性' 等输入参数
    """
    url = f"http://{CONFIG['node_ip']}:{CONFIG['port']}/appforge/openapi/v1/InvokeApp/{CONFIG['app_id']}"
    headers = {
        "Authorization": f"Bearer {CONFIG['api_key']}",
        "Content-Type": "application/json",
        "Accept": "*/*",
        "Connection": "keep-alive",
    }
    # 构造用户查询：将输入参数格式化后附加在系统提示词后
    user_query_content = f"""
请针对以下资产生成 TARA 分析 JSON：
- 功能：{asset_info.get('功能', '未知')}
- 资产：{asset_info.get('资产', '未知')}
- 资产类别：{asset_info.get('资产类别', '部件')}
- 安全属性：{asset_info.get('安全属性', '完整性')}
请直接输出 JSON 对象。
"""
    # 最终发送的 payload
    payload = {
        "id": CONFIG['app_id'],
        "query": user_query_content,
        # 如果平台支持在请求体中传入 system_prompt，请在此处添加
        # "system_prompt": SYSTEM_PROMPT 
    }
    # 注意：如果平台不支持单独传 system_prompt，则需要将其拼在 query 前
    # 这里假设智能体已经配置好了系统提示词，或者我们需要全量传入
    # 如果智能体是“白纸”状态，请使用下面的 full_query
    full_query = SYSTEM_PROMPT + "\n" + user_query_content
    payload["query"] = full_query
    result = {
        "input": asset_info,
        "output": None,
        "raw_text": "",
        "error": None,
        "status_code": None
    }
    try:
        response = session.post(url, headers=headers, json=payload, stream=True, timeout=CONFIG['timeout'])
        result["status_code"] = response.status_code
        if response.status_code != 200:
            result["error"] = f"HTTP {response.status_code}: {response.text[:200]}"
            return result
        # 解析 SSE 流
        answer_text = ""
        for raw_line in response.iter_lines():
            if not raw_line:
                continue
            decoded = raw_line.decode("utf-8")
            event = parse_appforge_sse_line(decoded)
            data = event.get("data")
            if not data:
                continue
            if data.get("type") == "answer":
                answer_text += data.get("content", "")
        result["raw_text"] = answer_text
        # 提取 JSON
        json_obj = extract_json_from_text(answer_text)
        if json_obj:
            result["output"] = json_obj
        else:
            result["error"] = "无法从回答中解析出有效 JSON"
    except Exception as e:
        result["error"] = str(e)
    return result
# ================= 4. 主流程控制 =================
def run_batch_generation(asset_list: List[Dict[str, str]]):
    """批量执行生成任务"""
    results = []
    total = len(asset_list)
    print(f">>> 开始批量生成，共 {total} 条任务，并发数: {CONFIG['max_workers']}")
    # 使用 Session 保持连接，提高效率
    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
            # 提交所有任务
            future_to_asset = {
                executor.submit(invoke_tara_agent, asset, session): asset 
                for asset in asset_list
            }
            # 处理完成的任务
            for idx, future in enumerate(as_completed(future_to_asset), 1):
                asset = future_to_asset[future]
                try:
                    res = future.result()
                    results.append(res)
                    # 实时打印进度
                    status_icon = "✅" if res["output"] else "❌"
                    print(f"[{idx}/{total}] {status_icon} 资产: {asset.get('资产')} | 耗时: N/A")
                    if res["error"]:
                        print(f"    错误: {res['error']}")
                    elif res["output"]:
                        # 打印部分关键结果预览
                        print(f"    损害场景: {res['output'].get('损害场景', '')[:30]}...")
                except Exception as e:
                    print(f"[{idx}/{total}] ❌ 任务执行异常: {asset.get('资产')}, {e}")
                    results.append({
                        "input": asset,
                        "output": None,
                        "error": str(e)
                    })
    return results
# ================= 5. 数据输入与执行入口 =================
if __name__ == "__main__":
    # 示例输入数据（你可以从 Excel/CSV 读取）
    # 注意：这里假设输入包含基础信息，输出由大模型补全
    input_assets = [
        {"功能": "自动紧急制动", "资产": "AEB ECU", "资产类别": "部件", "安全属性": "完整性"},
        {"功能": "车载娱乐系统", "资产": "用户隐私数据", "资产类别": "数据", "安全属性": "机密性"},
        {"功能": "远程控制", "资产": "TGW", "资产类别": "部件", "安全属性": "可用性"},
        {"功能": "车门控制", "资产": "门锁信号", "资产类别": "信号", "安全属性": "真实性"},
    ]
    # 执行批量生成
    final_results = run_batch_generation(input_assets)
    # 保存结果到 Excel
    output_data = []
    for res in final_results:
        if res["output"]:
            # 将输入字段与输出字段合并
            row = res["input"].copy()
            row.update(res["output"])
            output_data.append(row)
        else:
            # 记录失败项
            row = res["input"].copy()
            row["错误信息"] = res["error"]
            output_data.append(row)
    if output_data:
        df = pd.DataFrame(output_data)
        output_file = "TARA_生成结果.xlsx"
        df.to_excel(output_file, index=False)
        print(f"\n>>> 全部完成！结果已保存至: {output_file}")
    else:
        print("\n>>> 未生成有效结果。")

>>> 开始批量生成，共 4 条任务，并发数: 5
[1/4] ✅ 资产: 门锁信号 | 耗时: N/A
    损害场景: ...
[2/4] ✅ 资产: 用户隐私数据 | 耗时: N/A
    损害场景: ...
[3/4] ✅ 资产: AEB ECU | 耗时: N/A
    损害场景: ...
[4/4] ✅ 资产: TGW | 耗时: N/A
    损害场景: ...

>>> 全部完成！结果已保存至: TARA_生成结果.xlsx


In [26]:
import requests
import json
import time
import re
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Any, Optional
import pandas as pd
# ================= 1. 配置区域 =================
CONFIG = {
    "node_ip": "ai.catarc.ac.cn",
    "port": "9080",
    "app_id": "app-ewb874zb",
    "api_key": "75f8f1988e58589446f29492680dcc78",  # 请替换真实 API Key
    "timeout": 90,
    "max_workers": 5   # 并发数
}
# ================= 2. 提示词模板 =================
SYSTEM_PROMPT = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 标准。你的任务是根据给定的资产或系统上下文，执行威胁分析和风险评估（TARA），输出损害场景（危害场景）、威胁场景和攻击路径。
### 核心约束条件
1. **定义合规性**：所有输出必须严格遵循以下 ISO 21434 定义：
   - **损害场景**：必须包含[相关项功能与不良后果的关系]、[对道路使用者的危害]、[相关资产]。
   - **威胁场景**：必须包含[目标资产]、[资产被破坏的信息安全属性（保密性/完整性/可用性）]、[破坏原因/攻击意图]。
   - **攻击路径**：必须包含逻辑连贯的步骤，展示如何从攻击面到达目标资产以实现威胁场景，必须明确攻击的起点（如蜂窝网络、蓝牙、OBD、USB、物理接口）。
2. **一致性**：威胁场景必须能直接导致危害场景；攻击路径必须能实现威胁场景。
3. **格式要求**：直接输出最后的 JSON 格式结构化结果，不要输出思考过程。
### 思维链指引
在生成最终结果之前，先进行以下逐步思考：
1. **资产分析**：分析输入中提到的资产及其关键信息安全属性（CIA）。
2. **危害推演**：如果该资产的信息安全属性被破坏，会对道路使用者造成什么具体的危害？。
3. **威胁构建**：谁（攻击者）可能利用什么弱点来破坏该资产的属性？构建具体的威胁场景描述。
4. **路径分析**：攻击者如何进入系统？需要经过哪些组件或层级？。详细拆解攻击步骤。
### 必须遵守的标准与定义（严格执行）
#### 1. 资产类别与安全属性（严格遵循）
- **资产类别 = 「数据」**：安全属性限定为：完整性、机密性、可用性。
- **资产类别 = 「信号」**：安全属性限定为：完整性、机密性、真实性、可用性。
- **资产类别 = 「部件」**：安全属性限定为：完整性、机密性、真实性、不可抵赖性、权限属性、可用性。
#### 2. 威胁类型与安全属性映射（严格遵循）
- **真实性** → 威胁类型为「欺骗」
- **完整性** → 威胁类型为「篡改」
- **不可抵赖性** → 威胁类型为「否认」
- **机密性** → 威胁类型为「信息泄露」
- **可用性** → 威胁类型为「拒绝服务」
- **权限属性** → 威胁类型为「提权」
### 资产、安全属性、损害场景、威胁场景、攻击路径关系示例
一个危害场景能对应多个威胁场景,一个威胁场景能导致多个危害场景。
攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径;
如果部分攻击路径不会导致威胁场景的实现,则可能停止对该部分攻击路径的分析。
示例1:资产是存储在信息娱乐系统中的个人信息(客户个人偏好),该资产的信息安全属性为保密性。损害场景是由于该资产失去保密性,在未经客户同意的情况下,披露客户个人信息。
示例2:资产是制动功能的通信数据,该资产的信息安全属性是完整性。危害场景是车辆高速行驶时,因非预期的全力制动而与跟随车辆发生碰撞(追尾碰撞)。
示例3:威胁场景:欺骗制动 ECU 的 CAN 消息,导致 CAN 消息的完整性缺失,从而导致制动功能的完整性缺失。
———实现上述威胁场景的攻击路径:
1) 利用蜂窝接口损害远程通信 ECU;
2) 利用远程通信 ECU 的 CAN 通信损害网关 ECU;
3) 网关 ECU 转发恶意制动请求信号(非预期的快速减速)
## 种子示例（仅作样式参考，不要逐字复制）
示例 1（改编自附录 H 前照灯相关条目）：
{
  "功能": "自动远光灯",
  "资产": "前照灯请求信号",
  "资产类别": "信号",
  "安全属性": "完整性",
  "损害场景": "车辆夜间行驶时，前照灯意外关闭，导致驾驶员在无照明条件下高速行驶，存在与静止障碍物正面碰撞的风险。",
  "威胁场景": "攻击者篡改从电源开关执行器到“前灯请求”的信号完整性，导致前照灯被意外关闭。",
  "攻击路径": "1) 通过蜂窝网络接口入侵远程通信ECU；2) 被入侵的远程通信ECU发送恶意控制信号；3) 网关ECU转发恶意控制信号到电源开关执行器；4) 恶意信号伪装成“前灯关闭”请求。"}
示例 2（改编自 BSD 控制条目）：
{
  "功能": "盲区监测",
  "资产": "BSD ECU",
  "资产类别": "部件",
  "安全属性": "完整性",
  "损害场景": "BSD功能在变道或并线场景下因异常行为失效，导致驾驶员未收到盲区警告，存在侧面碰撞风险。",
  "威胁场景": "攻击者对BSD中的相关代码或数据非法篡改，导致数据或代码缺失，完整性被侵害，造成BSD控制无法使用，影响车辆操作。",
  "攻击路径": "1) 通过蜂窝网络接口入侵TGW；2) TGW通过CAN发送恶意信号至BSD；3) 攻击者检索BSD相关代码或数据；4) 对代码或数据进行恶意篡改或破坏。"}
### 当前任务要求
- 每次仅输出一个 JSON 对象，不要有多余文字。
- 生成的样本应在风格、用词、结构与「种子示例」保持高度一致。
"""
# ================= 3. 核心处理函数 =================
def parse_appforge_sse_line(raw_line: str) -> Dict[str, Any]:
    """解析 SSE 行"""
    line = raw_line.strip()
    if not line or line.startswith(":"): return {}
    if line.startswith("data:"):
        try: return {"data": json.loads(line[len("data:"):].strip())}
        except: return {}
    return {}
def clean_knowledge_tags(text: str) -> str:
    """
    删除知识库引用标签，如：
    <sup score="0.32..." ...>6</sup>
    """
    if not text: return ""
    # 正则匹配 <sup ...>数字</sup> 并替换为空
    clean_text = re.sub(r'<sup[^>]*>.*?</sup>', '', text, flags=re.DOTALL)
    return clean_text.strip()
def format_attack_path(path_data: Any) -> str:
    """
    将列表格式的攻击路径转换为编号文本
    输入: "['步骤1', '步骤2']" 或 ['步骤1', '步骤2']
    输出: "1.步骤1\n2.步骤2"
    """
    if not path_data:
        return ""
    path_list = []
    # 1. 如果已经是列表
    if isinstance(path_data, list):
        path_list = path_data
    # 2. 如果是字符串形式的列表 "['a', 'b']"
    elif isinstance(path_data, str):
        try:
            # 尝试用 ast.literal_eval 安全解析字符串列表
            evaluated = ast.literal_eval(path_data)
            if isinstance(evaluated, list):
                path_list = evaluated
            else:
                # 如果解析出来不是列表（比如就是普通字符串），直接返回
                return path_data
        except:
            # 解析失败（可能就是普通文本），直接返回原文本
            return path_data
    # 格式化输出
    formatted_lines = []
    for idx, step in enumerate(path_list, 1):
        # 去除步骤内部可能存在的多余标点
        step_text = str(step).strip()
        formatted_lines.append(f"{idx}.{step_text}")
    return "\n".join(formatted_lines)
def extract_json_from_text(text: str) -> Optional[Dict]:
    """从文本中提取 JSON"""
    if not text: return None
    try: return json.loads(text)
    except: pass
    # 提取 ```json ... ```
    match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    # 提取纯花括号
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return None
def invoke_tara_agent(asset_info: Dict[str, str], session: requests.Session) -> Dict[str, Any]:
    url = f"http://{CONFIG['node_ip']}:{CONFIG['port']}/appforge/openapi/v1/InvokeApp/{CONFIG['app_id']}"
    headers = {
        "Authorization": f"Bearer {CONFIG['api_key']}",
        "Content-Type": "application/json",
    }
    user_query = f"""
请针对以下资产生成 TARA 分析 JSON：
- 功能：{asset_info.get('功能', '未知')}
- 资产：{asset_info.get('资产', '未知')}
- 资产类别：{asset_info.get('资产类别', '部件')}
- 安全属性：{asset_info.get('安全属性', '完整性')}
"""
    # 构造请求体
    # 注意：尝试添加 retrieval_top_k=0 或类似参数尝试关闭检索
    # 如果平台不支持参数关闭，则依赖后处理清洗
    payload = {
        "id": CONFIG['app_id'],
        "query": SYSTEM_PROMPT + "\n" + user_query,
        # 尝试关闭知识库检索的参数（具体参数名需视平台文档而定，常用的有：
        # "use_knowledge": False, 
        # "retrieval_config": {"top_k": 0}
        # 此处保留请求，主要依赖清洗函数确保结果干净
    }
    result = {
        "input": asset_info,
        "output": None,
        "raw_text": "",
        "error": None
    }
    try:
        response = session.post(url, headers=headers, json=payload, stream=True, timeout=CONFIG['timeout'])
        if response.status_code != 200:
            result["error"] = f"HTTP {response.status_code}"
            return result
        answer_text = ""
        for raw_line in response.iter_lines():
            if not raw_line: continue
            decoded = raw_line.decode("utf-8")
            event = parse_appforge_sse_line(decoded)
            data = event.get("data")
            if data and data.get("type") == "answer":
                answer_text += data.get("content", "")
        # 关键步骤：清洗知识库标签
        answer_text = clean_knowledge_tags(answer_text)
        result["raw_text"] = answer_text
        json_obj = extract_json_from_text(answer_text)
        if json_obj:
            # 关键步骤：格式化攻击路径
            if "攻击路径" in json_obj:
                json_obj["攻击路径"] = format_attack_path(json_obj["攻击路径"])
            result["output"] = json_obj
        else:
            result["error"] = "JSON解析失败"
    except Exception as e:
        result["error"] = str(e)
    return result
# ================= 4. 主流程 =================
def run_batch_generation(asset_list: List[Dict[str, str]]):
    results = []
    total = len(asset_list)
    print(f">>> 开始批量生成，共 {total} 条任务")
    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
            futures = {executor.submit(invoke_tara_agent, asset, session): asset for asset in asset_list}
            for idx, future in enumerate(as_completed(futures), 1):
                res = future.result()
                results.append(res)
                status = "✅" if res["output"] else "❌"
                print(f"[{idx}/{total}] {status} {res['input'].get('资产')}")
    return results
# ================= 5. 执行入口 =================
if __name__ == "__main__":
    # 示例输入
    input_assets = [
        {"功能": "车门控制", "资产": "门锁信号", "资产类别": "信号", "安全属性": "真实性"},
        {"功能": "远程通信", "资产": "T-BOX", "资产类别": "部件", "安全属性": "完整性"},
    ]
    # 1. 执行生成
    raw_results = run_batch_generation(input_assets)
    # 2. 准备数据用于 Excel (格式化后)
    excel_rows = []
    # 3. 准备数据用于 JSON (保留结构)
    json_output_data = []
    for res in raw_results:
        if res["output"]:
            # 构建 Excel 行 (平铺数据)
            row = res["input"].copy()
            row.update(res["output"])
            excel_rows.append(row)
            # 构建 JSON 对象
            json_output_data.append(res["output"])
        else:
            # 记录错误
            excel_rows.append({**res["input"], "错误": res["error"]})
    # 4. 保存 Excel
    if excel_rows:
        df = pd.DataFrame(excel_rows)
        excel_file = "TARA_结果报表.xlsx"
        # 自动调整列宽
        writer = pd.ExcelWriter(excel_file, engine='openpyxl')
        df.to_excel(writer, index=False, sheet_name='TARA分析')
        # 简单调整列宽逻辑
        worksheet = writer.sheets['TARA分析']
        for col in worksheet.columns:
            max_length = 0
            column = col[0].column_letter
            for cell in col:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = (min(max_length, 60) + 2)
            worksheet.column_dimensions[column].width = adjusted_width
        writer.close()
        print(f">>> Excel 已保存: {excel_file}")
    # 5. 保存 JSON
    if json_output_data:
        json_file = "TARA_结果原始数据.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(json_output_data, f, ensure_ascii=False, indent=4)
        print(f">>> JSON 已保存: {json_file}")

>>> 开始批量生成，共 2 条任务
[1/2] ✅ 门锁信号
[2/2] ✅ T-BOX
>>> Excel 已保存: TARA_结果报表.xlsx
>>> JSON 已保存: TARA_结果原始数据.json


In [28]:
import requests
import json
import time
import re
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Any, Optional
import pandas as pd
# ================= 1. 配置区域 =================
CONFIG = {
    "node_ip": "ai.catarc.ac.cn",
    "port": "9080",
    "app_id": "app-ewb874zb",
    "api_key": "75f8f1988e58589446f29492680dcc78",  # 请替换真实 API Key
    "timeout": 90,
    "max_workers": 3   # 并发数建议适中，避免生成内容重复
}
# ================= 2. 提示词模板 =================
# 修改后的 Prompt：明确要求输出完整字段，且不需要用户输入具体资产
SYSTEM_PROMPT = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 标准。
你的任务是根据给定的领域提示，自主构想一个具体的资产，并执行威胁分析和风险评估（TARA）。
### 核心约束条件
1. **输出字段完整性**：输出的 JSON 必须严格包含以下 7 个字段：
   - "功能": (string) 资产所属的功能系统名称
   - "资产": (string) 具体的资产名称
   - "资产类别": (string) 必须是 "数据"、"信号"、"部件" 之一
   - "安全属性": (string) 根据资产类别选择对应的属性
   - "损害场景": (string) 必须包含[相关项功能与不良后果的关系]、[对道路使用者的危害]、[相关资产]。
   - "威胁场景": (string) 必须包含[目标资产]、[资产被破坏的信息安全属性（保密性/完整性/可用性）]、[破坏原因/攻击意图]。
   - "攻击路径": (list 或 string) 必须包含逻辑连贯的步骤，展示如何从攻击面到达目标资产以实现威胁场景，必须明确攻击的起点（如蜂窝网络、蓝牙、OBD、USB、物理接口）。
2. **资产类别与安全属性映射**：
- **资产类别 = 「数据」**：安全属性限定为：完整性、机密性、可用性。
- **资产类别 = 「信号」**：安全属性限定为：完整性、机密性、真实性、可用性。
- **资产类别 = 「部件」**：安全属性限定为：完整性、机密性、真实性、不可抵赖性、权限属性、可用性。
3. **自主性**：你需要根据提示的领域，自主构思一个合理的资产，不要询问用户。
4. **格式要求**：
   - 直接输出 JSON，不要有废话。
   - 输出内容严禁包含引用标记（如 <sup>...</sup>）。
5. **一致性**：威胁场景必须能直接导致危害场景；攻击路径必须能实现威胁场景。
### 示例输出格式
{
    "功能": "自动远光灯",
    "资产": "前照灯请求信号",
    "资产类别": "信号",
    "安全属性": "完整性",
    "损害场景": "车辆夜间行驶时...",
    "威胁场景": "攻击者篡改...",
    "攻击路径": "1.步骤1/n2.步骤2/n3.步骤3"
}
### 思维链指引
在生成最终结果之前，先进行以下逐步思考：
1. **资产分析**：分析输入中提到的资产及其关键信息安全属性（CIA）。
2. **危害推演**：如果该资产的信息安全属性被破坏，会对道路使用者造成什么具体的危害？。
3. **威胁构建**：谁（攻击者）可能利用什么弱点来破坏该资产的属性？构建具体的威胁场景描述。
4. **路径分析**：攻击者如何进入系统？需要经过哪些组件或层级？。详细拆解攻击步骤。
### 必须遵守的标准与定义（严格执行）
#### 1. 威胁类型与安全属性映射（严格遵循）
- **真实性** → 威胁类型为「欺骗」
- **完整性** → 威胁类型为「篡改」
- **不可抵赖性** → 威胁类型为「否认」
- **机密性** → 威胁类型为「信息泄露」
- **可用性** → 威胁类型为「拒绝服务」
- **权限属性** → 威胁类型为「提权」
### 资产、安全属性、损害场景、威胁场景、攻击路径关系示例
一个危害场景能对应多个威胁场景,一个威胁场景能导致多个危害场景。
攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径;
如果部分攻击路径不会导致威胁场景的实现,则可能停止对该部分攻击路径的分析。
示例1:资产是存储在信息娱乐系统中的个人信息(客户个人偏好),该资产的信息安全属性为保密性。损害场景是由于该资产失去保密性,在未经客户同意的情况下,披露客户个人信息。
示例2:资产是制动功能的通信数据,该资产的信息安全属性是完整性。危害场景是车辆高速行驶时,因非预期的全力制动而与跟随车辆发生碰撞(追尾碰撞)。
示例3:威胁场景:欺骗制动 ECU 的 CAN 消息,导致 CAN 消息的完整性缺失,从而导致制动功能的完整性缺失。
———实现上述威胁场景的攻击路径:
1) 利用蜂窝接口损害远程通信 ECU;
2) 利用远程通信 ECU 的 CAN 通信损害网关 ECU;
3) 网关 ECU 转发恶意制动请求信号(非预期的快速减速)
## 种子示例（仅作样式参考，不要逐字复制）
示例 1（改编自附录 H 前照灯相关条目）：
{
  "功能": "自动远光灯",
  "资产": "前照灯请求信号",
  "资产类别": "信号",
  "安全属性": "完整性",
  "损害场景": "车辆夜间行驶时，前照灯意外关闭，导致驾驶员在无照明条件下高速行驶，存在与静止障碍物正面碰撞的风险。",
  "威胁场景": "攻击者篡改从电源开关执行器到“前灯请求”的信号完整性，导致前照灯被意外关闭。",
  "攻击路径": "1.通过蜂窝网络接口入侵远程通信ECU
  2.被入侵的远程通信ECU发送恶意控制信号
  3.网关ECU转发恶意控制信号到电源开关执行器
  4.恶意信号伪装成“前灯关闭”请求"}
示例 2（改编自 BSD 控制条目）：
{
  "功能": "盲区监测",
  "资产": "BSD",
  "资产类别": "部件",
  "安全属性": "完整性",
  "损害场景": "BSD功能在变道或并线场景下因异常行为失效，导致驾驶员未收到盲区警告，存在侧面碰撞风险。",
  "威胁场景": "攻击者对BSD中的相关代码或数据非法篡改，导致数据或代码缺失，完整性被侵害，造成BSD控制无法使用，影响车辆操作。",
  "攻击路径": "1.通过蜂窝网络接口入侵TGW
  2.TGW通过CAN发送恶意信号至BSD
  3.攻击者检索BSD相关代码或数据
  4.对代码或数据进行恶意篡改或破坏"}
### 当前任务要求
- 生成的样本应在风格、用词、结构与「种子示例」保持高度一致。
"""
# ================= 3. 核心处理函数 =================
def parse_appforge_sse_line(raw_line: str) -> Dict[str, Any]:
    """解析 SSE 行"""
    line = raw_line.strip()
    if not line or line.startswith(":"): return {}
    if line.startswith("data:"):
        try: return {"data": json.loads(line[len("data:"):].strip())}
        except: return {}
    return {}
def clean_knowledge_tags(text: str) -> str:
    """删除知识库引用标签"""
    if not text: return ""
    return re.sub(r'<sup[^>]*>.*?</sup>', '', text, flags=re.DOTALL | re.IGNORECASE).strip()
def format_attack_path(path_data: Any) -> str:
    """
    强制将攻击路径转换为编号格式
    输入: "['步骤1。', '步骤2。']" 或 ['步骤1', '步骤2']
    输出: "1.步骤1\n2.步骤2"
    """
    if not path_data: return ""
    path_list = []
    # 1. 处理列表对象
    if isinstance(path_data, list):
        path_list = path_data
    # 2. 处理字符串（可能是字符串形式的列表）
    elif isinstance(path_data, str):
        try:
            # 尝试解析字符串列表
            evaluated = ast.literal_eval(path_data)
            if isinstance(evaluated, list):
                path_list = evaluated
            else:
                # 如果解析出来不是列表，说明可能已经是纯文本了，直接返回
                return path_data
        except:
            # 解析失败，直接返回原字符串
            return path_data
    # 格式化输出
    formatted_lines = []
    for idx, step in enumerate(path_list, 1):
        step_text = str(step).strip()
        # 去除末尾的句号，保持格式统一
        if step_text.endswith('。'): step_text = step_text[:-1]
        formatted_lines.append(f"{idx}.{step_text}")
    return "\n".join(formatted_lines)
def extract_json_from_text(text: str) -> Optional[Dict]:
    """从文本中提取 JSON，兼容单引号格式"""
    if not text: return None
    # 尝试直接解析
    try: return json.loads(text)
    except: pass
    # 尝试提取代码块
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    # 尝试提取花括号内容
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        json_str = match.group(0)
        try: 
            return json.loads(json_str)
        except json.JSONDecodeError:
            # 关键：如果标准 JSON 解析失败（可能因为单引号），尝试 ast.literal_eval
            try:
                return ast.literal_eval(json_str)
            except:
                pass
    return None
def invoke_tara_agent(task_hint: str, session: requests.Session) -> Dict[str, Any]:
    """调用智能体生成 TARA"""
    url = f"http://{CONFIG['node_ip']}:{CONFIG['port']}/appforge/openapi/v1/InvokeApp/{CONFIG['app_id']}"
    headers = {
        "Authorization": f"Bearer {CONFIG['api_key']}",
        "Content-Type": "application/json",
    }
    # 构造用户提示：仅提供领域提示，不提供具体资产
    user_query = f"请针对【{task_hint}】相关的车辆资产，生成一个 TARA 分析 JSON 样本。请自主选择具体资产并补全所有字段。"
    payload = {
        "id": CONFIG['app_id'],
        "query": SYSTEM_PROMPT + "\n" + user_query,
    }
    result = {
        "input_hint": task_hint,
        "output": None,
        "raw_text": "",
        "error": None
    }
    try:
        response = session.post(url, headers=headers, json=payload, stream=True, timeout=CONFIG['timeout'])
        if response.status_code != 200:
            result["error"] = f"HTTP {response.status_code}"
            return result
        answer_text = ""
        for raw_line in response.iter_lines():
            if not raw_line: continue
            decoded = raw_line.decode("utf-8")
            event = parse_appforge_sse_line(decoded)
            data = event.get("data")
            if data and data.get("type") == "answer":
                answer_text += clean_knowledge_tags(data.get("content", ""))
        result["raw_text"] = answer_text
        json_obj = extract_json_from_text(answer_text)
        if json_obj:
            # 强制格式化攻击路径
            if "攻击路径" in json_obj:
                json_obj["攻击路径"] = format_attack_path(json_obj["攻击路径"])
            result["output"] = json_obj
        else:
            result["error"] = "JSON解析失败"
    except Exception as e:
        result["error"] = str(e)
    return result
# ================= 4. 主流程 =================
def run_batch_generation(task_hints: List[str]):
    results = []
    total = len(task_hints)
    print(f">>> 开始批量生成，共 {total} 条任务")
    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
            futures = {executor.submit(invoke_tara_agent, hint, session): hint for hint in task_hints}
            for idx, future in enumerate(as_completed(futures), 1):
                res = future.result()
                results.append(res)
                status = "✅" if res["output"] else "❌"
                print(f"[{idx}/{total}] {status} 提示: {res['input_hint']}")
    return results
# ================= 5. 执行入口 =================
if __name__ == "__main__":
    # 输入：提供一些领域关键词，引导智能体生成多样化的内容
    # 智能体会根据这些提示，自动构想具体的资产、功能等
    input_hints = [
        "智能座舱与车窗控制",
        "制动系统与行车安全",
        "动力系统与电池管理",
        "远程通信与T-BOX",
        "网关与数据转发",
        "自动驾驶与传感器"
    ]
    # 执行生成
    raw_results = run_batch_generation(input_hints)
    # 准备数据
    excel_rows = []
    json_output_data = []
    for res in raw_results:
        if res["output"]:
            # 输出包含所有字段，直接使用
            excel_rows.append(res["output"])
            json_output_data.append(res["output"])
        else:
            excel_rows.append({"输入提示": res["input_hint"], "错误": res["error"]})
    # 保存 Excel
    if excel_rows:
        df = pd.DataFrame(excel_rows)
        # 调整列顺序，确保关键字段在前
        cols = ["功能", "资产", "资产类别", "安全属性", "损害场景", "威胁场景", "攻击路径", "错误"]
        # 过滤掉不存在的列
        existing_cols = [c for c in cols if c in df.columns]
        df = df[existing_cols]
        excel_file = "TARA_自动生成报表.xlsx"
        df.to_excel(excel_file, index=False)
        print(f">>> Excel 已保存: {excel_file}")
    # 保存 JSON
    if json_output_data:
        json_file = "TARA_自动生成数据.json"
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(json_output_data, f, ensure_ascii=False, indent=4)
        print(f">>> JSON 已保存: {json_file}")

>>> 开始批量生成，共 6 条任务
[1/6] ✅ 提示: 智能座舱与车窗控制
[2/6] ✅ 提示: 制动系统与行车安全
[3/6] ✅ 提示: 动力系统与电池管理
[4/6] ✅ 提示: 自动驾驶与传感器
[5/6] ✅ 提示: 远程通信与T-BOX
[6/6] ✅ 提示: 网关与数据转发
>>> Excel 已保存: TARA_自动生成报表.xlsx
>>> JSON 已保存: TARA_自动生成数据.json


In [ ]:
import requests
import json
import time
import re
import ast
import os
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Any, Optional
import pandas as pd
from datetime import datetime
# ================= 1. 配置区域 =================
CONFIG = {
    "node_ip": "ai.catarc.ac.cn",
    "port": "9080",
    "app_id": "app-ewb874zb",
    "api_key": "key",  # 请替换真实 API Key
    "timeout": 120,
    "max_workers": 10,
    "batch_size": 50,
    "max_retries": 2,
    "retry_delay": 3,
    "target_count": 10000,
}
# 文件路径
OUTPUT_DIR = "tara_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EXCEL_FILE = os.path.join(OUTPUT_DIR, "TARA_自动生成结果.xlsx")
JSON_FILE = os.path.join(OUTPUT_DIR, "TARA_自动生成结果.json")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
# ================= 2. 提示词模板 =================
SYSTEM_PROMPT = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 标准。
你的任务是**完全自主地**生成一条 TARA（威胁分析与风险评估）样本数据。
### 核心要求
1. **完全自主选择**：你需要根据给定的【随机种子】，自主选择：
   - 一个具体的车辆功能（如：自动紧急制动、远程解锁等）
   - 该功能相关的一个具体资产（如：CAN信号、ECU、传感器数据等）
   - 资产类别（数据/信号/部件）
   - 适合该资产的安全属性
2. **多样性要求**：每条数据应该覆盖不同的功能、资产和场景，避免重复。
3. **输出格式**：直接输出一个 JSON 对象，包含以下字段：
   ```json
   {
       "功能": "xxx",
       "资产": "xxx",
       "资产类别": "数据/信号/部件",
       "安全属性": "xxx",
       "损害场景": "xxx",
       "威胁场景": "xxx",
       "攻击路径": ["步骤1", "步骤2", ...]
   }
资产类别与安全属性映射（必须严格遵守）
数据：完整性、机密性、可用性
信号：完整性、机密性、真实性、可用性
部件：完整性、机密性、真实性、不可抵赖性、权限属性、可用性
场景定义（ISO 21434 严格遵循）
损害场景：必须包含 [相关项功能与不良后果的关系] + [对道路使用者的危害] + [相关资产]
威胁场景：必须包含 [目标资产] + [被破坏的安全属性] + [破坏原因/攻击意图]
攻击路径：必须展示从攻击面（蜂窝/蓝牙/OBD/USB等）到目标资产的逻辑连贯步骤
格式约束
直接输出 JSON，不要有任何解释文字
禁止包含引用标记（如 <sup>...</sup>）
攻击路径使用字符串列表格式
确保生成的场景真实可信，符合实际车辆架构
示例（仅供参考格式，请勿重复内容）
{
"功能": "自动远光灯",
"资产": "前照灯请求信号",
"资产类别": "信号",
"安全属性": "完整性",
"损害场景": "车辆夜间行驶时，前照灯意外关闭，导致驾驶员在无照明条件下高速行驶，存在与静止障碍物正面碰撞的风险。",
"威胁场景": "攻击者篡改从前照灯控制器到电源开关执行器的信号完整性，导致前照灯被意外关闭。",
"攻击路径": [
"通过蜂窝网络入侵远程通信ECU",
"利用远程通信ECU的CAN通信损害网关ECU",
"网关ECU转发恶意控制信号到前照灯控制器",
"前照灯控制器执行伪造的关闭指令"
]
}
"""
多样性提示词（用于引导生成不同领域的内容）
DOMAIN_HINTS = [
"智能驾驶辅助系统（ADAS）",
"动力与能源管理系统",
"车身控制与舒适系统",
"信息娱乐与座舱系统",
"远程通信与车联网服务",
"制动与安全系统",
"转向与悬架系统",
"充电与电池管理系统",
"网关与网络安全",
"传感器与数据采集系统",
"OTA升级与远程诊断",
"无钥匙进入与启动系统",
"环境感知与决策系统",
"车辆定位与导航系统",
"驾驶员监测与预警系统",
"热管理与温控系统",
"灯光与视野系统",
"被动安全与约束系统",
"数据存储与处理系统",
"人机交互与显示系统"
]
================= 3. 核心处理函数 =================
def parse_appforge_sse_line(raw_line: str) -> Dict[str, Any]:
"""解析 SSE 行"""
line = raw_line.strip()
if not line or line.startswith(":"): return {}
if line.startswith("data:"):
try: return {"data": json.loads(line[len("data:"):].strip())}
except: return {}
return {}
def clean_knowledge_tags(text: str) -> str:
"""删除知识库引用标签"""
if not text: return ""
return re.sub(r'<sup[^>]*>.*?</sup>', '', text, flags=re.DOTALL | re.IGNORECASE).strip()
def format_attack_path(path_data: Any) -> str:
"""格式化攻击路径为编号文本"""
if not path_data: return ""
path_list = []
if isinstance(path_data, list):
    path_list = path_data
elif isinstance(path_data, str):
    try:
        evaluated = ast.literal_eval(path_data)
        if isinstance(evaluated, list):
            path_list = evaluated
        else:
            return path_data
    except:
        return path_data
formatted_lines = []
for idx, step in enumerate(path_list, 1):
    step_text = str(step).strip()
    if step_text.endswith('。'): step_text = step_text[:-1]
    formatted_lines.append(f"{idx}.{step_text}")
return "\n".join(formatted_lines)
def extract_json_from_text(text: str) -> Optional[Dict]:
"""从文本中提取 JSON，兼容单引号格式"""
if not text: return None
try: return json.loads(text)
except: pass
match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
if match:
    try: return json.loads(match.group(1))
    except: pass
match = re.search(r'\{.*\}', text, re.DOTALL)
if match:
    json_str = match.group(0)
    try: 
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except:
            pass
return None
def invoke_tara_agent(random_seed: int, domain_hint: str, session: requests.Session, retry_count: int = 0) -> Dict[str, Any]:
"""调用智能体自主生成 TARA"""
url = f"http://{CONFIG['node_ip']}:{CONFIG['port']}/appforge/openapi/v1/InvokeApp/{CONFIG['app_id']}"
headers = {
"Authorization": f"Bearer {CONFIG['api_key']}",
"Content-Type": "application/json",
}
# 构造用户提示：只给随机种子和领域提示，让模型自主生成
user_query = f"""
【随机种子】：{random_seed}
【领域提示】：{domain_hint}
请根据上述信息，完全自主地生成一条 TARA 分析数据。
要求：
选择一个与【领域提示】相关的具体功能
选择该功能相关的一个具体资产
自主确定资产类别和安全属性
生成符合 ISO 21434 标准的损害场景、威胁场景和攻击路径
直接输出 JSON，不要有任何其他文字
"""
payload = {
"id": CONFIG['app_id'],
"query": SYSTEM_PROMPT + "\n" + user_query,
}
result = {
"seed": random_seed,
"domain": domain_hint,
"output": None,
"raw_text": "",
"error": None,
"retry_count": retry_count
}
try:
response = session.post(url, headers=headers, json=payload, stream=True, timeout=CONFIG['timeout'])
 if response.status_code == 429:
     result["error"] = "Rate Limited"
     return result
 if response.status_code != 200:
     result["error"] = f"HTTP {response.status_code}"
     return result
 answer_text = ""
 for raw_line in response.iter_lines():
     if not raw_line: continue
     decoded = raw_line.decode("utf-8")
     event = parse_appforge_sse_line(decoded)
     data = event.get("data")
     if data and data.get("type") == "answer":
         answer_text += clean_knowledge_tags(data.get("content", ""))
 result["raw_text"] = answer_text
 json_obj = extract_json_from_text(answer_text)
 if json_obj:
     if "攻击路径" in json_obj:
         json_obj["攻击路径"] = format_attack_path(json_obj["攻击路径"])
     # 添加随机种子标记（用于去重和追溯）
     json_obj["_seed"] = random_seed
     result["output"] = json_obj
 else:
     result["error"] = "JSON解析失败"
except requests.exceptions.Timeout:
result["error"] = "请求超时"
except Exception as e:
result["error"] = str(e)
return result
================= 4. 进度管理 =================
def load_progress() -> Dict[str, Any]:
"""加载进度"""
if os.path.exists(PROGRESS_FILE):
try:
with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
return json.load(f)
except:
pass
return {"completed": 0, "results": []}
def save_progress(progress: Dict[str, Any]):
"""保存进度"""
with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
json.dump(progress, f, ensure_ascii=False)
def save_results(results: List[Dict], append: bool = False):
"""保存结果到文件"""
if not results:
return
output_data = [r["output"] for r in results if r.get("output")]
if not output_data:
    return
# 保存 JSON
if append and os.path.exists(JSON_FILE):
    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    output_data = existing + output_data
# 去重（根据 _seed）
seen_seeds = set()
unique_data = []
for item in output_data:
    seed = item.get("_seed")
    if seed and seed not in seen_seeds:
        seen_seeds.add(seed)
        unique_data.append(item)
    elif not seed:
        unique_data.append(item)
with open(JSON_FILE, 'w', encoding='utf-8') as f:
    json.dump(unique_data, f, ensure_ascii=False, indent=2)
# 保存 Excel（移除 _seed 字段）
excel_data = []
for item in unique_data:
    excel_item = {k: v for k, v in item.items() if not k.startswith('_')}
    excel_data.append(excel_item)
df = pd.DataFrame(excel_data)
cols = ["功能", "资产", "资产类别", "安全属性", "损害场景", "威胁场景", "攻击路径"]
df = df[[c for c in cols if c in df.columns]]
df.to_excel(EXCEL_FILE, index=False)
================= 5. 主流程 =================
def run_batch_generation():
"""批量生成主流程"""
# 加载进度
progress = load_progress()
start_idx = progress.get("completed", 0)
total = CONFIG['target_count']
print(f"\n{'='*70}")
print(f"  TARA 自动批量生成任务")
print(f"{'='*70}")
print(f"  目标数量: {total}")
print(f"  已完成: {start_idx}")
print(f"  待生成: {total - start_idx}")
print(f"  并发数: {CONFIG['max_workers']}")
print(f"  批次大小: {CONFIG['batch_size']}")
print(f"  模式: 完全自主生成（无需输入）")
print(f"{'='*70}\n")
if start_idx >= total:
    print(">>> 任务已完成，无需重新生成")
    return
results = []
start_time = time.time()
# 设置随机种子
random.seed(int(time.time()))
with requests.Session() as session:
    for batch_start in range(start_idx, total, CONFIG['batch_size']):
        batch_end = min(batch_start + CONFIG['batch_size'], total)
        batch_results = []
        print(f"\n>>> 处理批次 {batch_start+1}-{batch_end}/{total}")
        # 为本批次生成随机种子和领域提示
        batch_tasks = []
        for i in range(batch_start, batch_end):
            seed = random.randint(1, 999999)
            domain = random.choice(DOMAIN_HINTS)
            batch_tasks.append((seed, domain))
        with ThreadPoolExecutor(max_workers=CONFIG['max_workers']) as executor:
            futures = {
                executor.submit(invoke_tara_agent, seed, domain, session): (seed, domain)
                for seed, domain in batch_tasks
            }
            for future in as_completed(futures):
                seed, domain = futures[future]
                try:
                    result = future.result()
                    # 失败重试
                    if result["error"] and result["retry_count"] < CONFIG['max_retries']:
                        if result["error"] == "Rate Limited":
                            time.sleep(CONFIG['retry_delay'])
                        result = invoke_tara_agent(
                            seed, 
                            domain, 
                            session, 
                            result["retry_count"] + 1
                        )
                    batch_results.append(result)
                    # 实时显示进度
                    if result["output"]:
                        output = result["output"]
                        print(f"  ✅ [{len(batch_results)}/{len(batch_tasks)}] "
                              f"功能: {output.get('功能', 'N/A')[:15]:15s} | "
                              f"资产: {output.get('资产', 'N/A')[:15]:15s}")
                    else:
                        print(f"  ❌ [{len(batch_results)}/{len(batch_tasks)}] 错误: {result['error']}")
                except Exception as e:
                    print(f"  ❌ 任务异常: {e}")
                    batch_results.append({
                        "seed": seed,
                        "domain": domain,
                        "error": str(e)
                    })
        # 保存批次结果
        results.extend(batch_results)
        # 更新进度
        progress["completed"] = batch_end
        progress["results"] = [r for r in results if r.get("output")]
        save_progress(progress)
        # 保存文件
        save_results([r for r in batch_results if r.get("output")], append=(batch_start > start_idx))
        # 计算统计信息
        elapsed = time.time() - start_time
        speed = (batch_end - start_idx) / elapsed if elapsed > 0 else 0
        remaining = (total - batch_end) / speed if speed > 0 else 0
        success_count = len([r for r in results if r.get("output")])
        success_rate = success_count / len(results) * 100 if results else 0
        print(f"\n  📊 统计信息:")
        print(f"     成功率: {success_rate:.1f}%")
        print(f"     已用时: {elapsed/60:.1f} 分钟")
        print(f"     预计剩余: {remaining/60:.1f} 分钟")
        print(f"     已保存: {JSON_FILE}")
        # 批次间休息
        if batch_end < total:
            time.sleep(2)
# 最终统计
final_results = [r for r in results if r.get("output")]
print(f"\n{'='*70}")
print(f"  ✅ 任务完成！")
print(f"{'='*70}")
print(f"  总计: {total}")
print(f"  成功: {len(final_results)}")
print(f"  失败: {len(results) - len(final_results)}")
print(f"  成功率: {len(final_results)/len(results)*100:.1f}%")
print(f"  结果文件:")
print(f"    - JSON: {JSON_FILE}")
print(f"    - Excel: {EXCEL_FILE}")
print(f"{'='*70}\n")
================= 6. 数据质量检查 =================
def check_data_quality():
"""检查生成数据的质量"""
if not os.path.exists(JSON_FILE):
print(">>> 未找到数据文件")
return
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"\n{'='*70}")
print(f"  数据质量检查")
print(f"{'='*70}")
print(f"  总数量: {len(data)}")
# 统计字段完整性
required_fields = ["功能", "资产", "资产类别", "安全属性", "损害场景", "威胁场景", "攻击路径"]
complete_count = 0
for item in data:
    if all(field in item and item[field] for field in required_fields):
        complete_count += 1
print(f"  完整记录: {complete_count}/{len(data)} ({complete_count/len(data)*100:.1f}%)")
# 统计资产类别分布
from collections import Counter
categories = Counter([item.get("资产类别", "未知") for item in data])
print(f"\n  资产类别分布:")
for cat, count in categories.most_common():
    print(f"    - {cat}: {count} ({count/len(data)*100:.1f}%)")
# 统计安全属性分布
attrs = Counter([item.get("安全属性", "未知") for item in data])
print(f"\n  安全属性分布:")
for attr, count in attrs.most_common():
    print(f"    - {attr}: {count} ({count/len(data)*100:.1f}%)")
print(f"{'='*70}\n")
================= 7. 执行入口 =================
if name == "main":
import sys
if len(sys.argv) > 1 and sys.argv[1] == "--check":
    # 数据质量检查
    check_data_quality()
else:
    # 运行批量生成
    run_batch_generation()
    # 生成完成后检查质量
    check_data_quality()

In [ ]:
# 1. 安装
pip install gradio openai

# 2. 核心代码示例（main.py）
import gradio as gr
from openai import OpenAI

client = OpenAI(base_url="你的模型API地址", api_key="你的密钥")

def chat(message, history):
    messages = [{"role": "system", "content": "你是一个AI助手"}]
    for h in history:
        messages.append({"role": "user" if h[0] else "assistant", "content": h[0] if h[0] else h[1]})
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(model="your-model", messages=messages, stream=True)
    partial = ""
    for chunk in response:
        if chunk.choices[0].delta.content:
            partial += chunk.choices[0].delta.content
            yield partial

gr.ChatInterface(chat, title="我的AI助手").launch()